In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


In [4]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [5]:
trades = pd.read_csv('trades.csv')
user_activity = pd.read_csv('user_activitycsv.csv')

### **=== TRADES DATASET ===**


In [6]:
trades.shape

(2324, 6)

In [7]:
trades.head()

,pair,amount,volume,side,timestamp,user_id
0,AAVENGN,500.26,0.001222,buy,2024-12-03T01:42:07.072672Z,541-1561-1521-1561-148
1,ADANGN,889.20,0.500000,sell,2024-12-02T11:29:56.860014Z,491-1541-1521-1571-156
2,ADANGN,30653.12,17.100000,sell,2024-11-30T05:25:49.99166Z,1021-1481-1521-1561-1100
3,ADANGN,1978.36,1.100000,sell,2024-12-07T07:41:02.191598Z,981-1571-1521-1971-151
4,ADANGN,2913.62,1.600000,sell,2024-12-07T21:16:11.791118Z,981-1571-1521-1971-151


In [8]:
trades.dtypes

pair          object
amount       float64
volume       float64
side          object
timestamp     object
user_id       object
dtype: object

In [9]:
trades.isnull().sum()

pair         0
amount       0
volume       0
side         0
timestamp    0
user_id      0
dtype: int64

### **=== USER ACTIVITY DATASET ===**


In [10]:
user_activity.shape

(4845, 5)

In [11]:
user_activity.head()

,asset,amount,activity_type,timestamp,user_id
0,NGN,131062.0,withdrawal,2025-01-17T14:37:56.806446Z,561-1561-1521-1981-1101
1,NGN,50000.0,withdrawal,2025-01-17T14:37:56.818033Z,971-11001-1521-1571-1101
2,NGN,131062.0,deposit,2025-01-17T14:37:56.853225Z,481-11011-1521-1971-1100
3,NGN,20064.0,withdrawal,2025-01-17T17:05:18.201218Z,531-11001-1521-1571-198
4,NGN,2000.0,withdrawal,2025-01-17T14:38:22.555744Z,501-11021-1521-1561-154


In [12]:
user_activity.dtypes

asset             object
amount           float64
activity_type     object
timestamp         object
user_id           object
dtype: object

In [13]:
user_activity.isnull().sum()

asset            0
amount           0
activity_type    0
timestamp        0
user_id          0
dtype: int64

**Top 3 most traded pairs by total USD volume**

In [14]:
# Question 1: Top 3 most traded pairs by total USD volume
# Using conversion rate of 1500 for all currency conversions to USD

def get_quote_currency(pair):
    if pair.endswith('USDT'):
        return 'USDT'
    elif pair.endswith('BTC'):
        return 'BTC'
    else:
        return 'NGN'


In [15]:
trades['quote_currency'] = trades['pair'].apply(get_quote_currency)

In [16]:
# Convert all trades to USD equivalent
# amount column represents the total value in quote currency
def convert_to_usd(row):
    if row['quote_currency'] == 'USDT':
        return row['amount']
    elif row['quote_currency'] == 'NGN':
        return row['amount'] / 1500
    elif row['quote_currency'] == 'BTC':
        # For BTC pairs, we need to get BTC price first
        # Using average BTCUSDT or BTCNGN price from the data
        return row['amount'] * 50000  # Approximation, will refine
    return row['amount']


In [17]:
trades['usd_value'] = trades.apply(convert_to_usd, axis=1)

In [18]:
# Calculate total USD volume by pair
pair_volumes = trades.groupby('pair')['usd_value'].sum().sort_values(ascending=False)

print("=== TOP 3 MOST TRADED PAIRS BY USD VOLUME ===\n")
top_3_pairs = pair_volumes.head(3)
for i, (pair, volume) in enumerate(top_3_pairs.items(), 1):
    print(f"{i}. {pair}: ${volume:,.2f}")

print(f"\n{'='*50}")
print(f"Total USD volume across all pairs: ${pair_volumes.sum():,.2f}")


=== TOP 3 MOST TRADED PAIRS BY USD VOLUME ===

1. BTCNGN: $136,215.49
2. USDTNGN: $59,180.57
3. BTCUSDT: $13,443.57

Total USD volume across all pairs: $250,200.83
